# Brain Tumor MRI Classification: CNN Baseline vs Transfer Learning

This notebook compares a custom CNN baseline with four ImageNet-pretrained models (EfficientNetB0, ResNet50, MobileNetV2, DenseNet121) on the Brain Tumor MRI dataset (glioma, meningioma, notumor, pituitary).

**Kaggle setup:** enable **Internet** in the notebook settings (Settings → Internet → On) so the pretrained ImageNet weights can be downloaded, and select a **GPU** accelerator.


## 1. Imports

In [ ]:
import os
import gc
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, ResNet50, MobileNetV2, DenseNet121
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

## 2. Environment Check

In [ ]:
print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 3. Configuration

In [ ]:
TRAIN_PATH = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training"
TEST_PATH = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing"
WORKING_DIR = "/kaggle/working"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
SEED = 42

HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 8
CNN_EPOCHS = 30
FINE_TUNE_LAYERS = 30

tf.keras.utils.set_random_seed(SEED)

print("Training exists:", os.path.exists(TRAIN_PATH))
print("Testing exists:", os.path.exists(TEST_PATH))

## 4. Dataset Preparation

In [ ]:
classes = sorted([f for f in os.listdir(TRAIN_PATH) if os.path.isdir(os.path.join(TRAIN_PATH, f))])
print("Classes:", classes)
print("Number of classes:", len(classes))

for class_name in classes:
    train_count = len(os.listdir(os.path.join(TRAIN_PATH, class_name)))
    test_count = len(os.listdir(os.path.join(TEST_PATH, class_name)))
    print(f"{class_name:12} | Train: {train_count:5} | Test: {test_count:5}")

In [ ]:
def create_dataframe(directory):
    data = []
    for class_name in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, class_name)
        if not os.path.isdir(class_path):
            continue
        for image_name in os.listdir(class_path):
            data.append({"image_path": os.path.join(class_path, image_name), "label": class_name})
    return pd.DataFrame(data)

train_df = create_dataframe(TRAIN_PATH)
test_df = create_dataframe(TEST_PATH)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=SEED
)

print("Train:", len(train_df), "| Validation:", len(val_df), "| Test:", len(test_df))

### Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (title, df) in zip(axes, [("Train", train_df), ("Validation", val_df), ("Test", test_df)]):
    sns.countplot(data=df, x="label", order=classes, ax=ax)
    ax.set_title(f"{title} Distribution")
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

### Sample Images

In [ ]:
plt.figure(figsize=(12, 8))
sample_df = train_df.sample(12, random_state=SEED)
for i, (_, row) in enumerate(sample_df.iterrows()):
    image = tf.keras.utils.load_img(row["image_path"], target_size=IMAGE_SIZE)
    plt.subplot(3, 4, i + 1)
    plt.imshow(tf.keras.utils.img_to_array(image).astype("uint8"))
    plt.title(row["label"])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Helper Functions

Shared utilities for data generators, callbacks, training, evaluation and plotting so every model is trained and evaluated in exactly the same way.

In [ ]:
def create_generators(preprocessing_function=None):
    train_datagen = ImageDataGenerator(
        preprocessing_function=preprocessing_function,
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True
    )
    eval_datagen = ImageDataGenerator(preprocessing_function=preprocessing_function)

    train_gen = train_datagen.flow_from_dataframe(
        dataframe=train_df, x_col="image_path", y_col="label",
        target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=True, seed=SEED
    )
    val_gen = eval_datagen.flow_from_dataframe(
        dataframe=val_df, x_col="image_path", y_col="label",
        target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=False
    )
    test_gen = eval_datagen.flow_from_dataframe(
        dataframe=test_df, x_col="image_path", y_col="label",
        target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=False
    )
    return train_gen, val_gen, test_gen


def make_callbacks(name, patience=4):
    return [
        EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7),
        ModelCheckpoint(
            os.path.join(WORKING_DIR, f"{name}_best.keras"),
            monitor="val_accuracy",
            save_best_only=True
        )
    ]


def merge_histories(*histories):
    merged = {}
    for h in histories:
        for key, values in h.history.items():
            merged.setdefault(key, []).extend(values)
    return merged


def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )


def evaluate_model(name, model, test_gen, history, train_time, fine_tune_start=None):
    test_gen.reset()
    probabilities = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)
    y_true = test_gen.classes
    class_names = list(test_gen.class_indices.keys())

    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    result = {
        "name": name,
        "history": history,
        "test_accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": report["macro avg"]["f1-score"],
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "per_class_f1": {c: report[c]["f1-score"] for c in class_names},
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "class_names": class_names,
        "parameters": model.count_params(),
        "train_time_min": train_time / 60,
        "best_val_accuracy": max(history["val_accuracy"]),
        "fine_tune_start": fine_tune_start
    }

    print(f"{name} Test Accuracy: {result['test_accuracy']:.4f}")
    print(classification_report(y_true, y_pred, target_names=class_names))
    return result


def plot_result(result):
    history = result["history"]
    epochs = range(1, len(history["accuracy"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    axes[0].plot(epochs, history["accuracy"], label="Train Accuracy")
    axes[0].plot(epochs, history["val_accuracy"], label="Validation Accuracy")
    axes[0].set_title(f"{result['name']} Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")

    axes[1].plot(epochs, history["loss"], label="Train Loss")
    axes[1].plot(epochs, history["val_loss"], label="Validation Loss")
    axes[1].set_title(f"{result['name']} Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")

    if result["fine_tune_start"] is not None:
        for ax in axes[:2]:
            ax.axvline(result["fine_tune_start"] + 0.5, color="gray", linestyle="--", label="Fine-tuning starts")
    for ax in axes[:2]:
        ax.legend()

    sns.heatmap(
        result["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
        xticklabels=result["class_names"], yticklabels=result["class_names"], ax=axes[2]
    )
    axes[2].set_title(f"{result['name']} Confusion Matrix (Test Acc: {result['test_accuracy']:.4f})")
    axes[2].set_xlabel("Predicted")
    axes[2].set_ylabel("Actual")

    plt.tight_layout()
    plt.show()


def build_transfer_model(base_class):
    base_model = base_class(include_top=False, weights="imagenet", input_shape=IMAGE_SIZE + (3,))
    base_model.trainable = False

    inputs = Input(shape=IMAGE_SIZE + (3,))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.5)(x)
    outputs = Dense(NUM_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    compile_model(model, 1e-3)
    return model, base_model


def run_transfer_experiment(name, base_class, preprocess_fn):
    train_gen, val_gen, test_gen = create_generators(preprocess_fn)
    model, base_model = build_transfer_model(base_class)

    start = time.time()
    head_history = model.fit(
        train_gen, validation_data=val_gen,
        epochs=HEAD_EPOCHS, callbacks=make_callbacks(name), verbose=1
    )
    head_epochs_run = len(head_history.history["loss"])

    base_model.trainable = True
    for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    compile_model(model, 1e-5)

    fine_history = model.fit(
        train_gen, validation_data=val_gen,
        epochs=FINE_TUNE_EPOCHS, callbacks=make_callbacks(name, patience=3), verbose=1
    )
    train_time = time.time() - start

    history = merge_histories(head_history, fine_history)
    result = evaluate_model(name, model, test_gen, history, train_time, fine_tune_start=head_epochs_run)

    del model, base_model, train_gen, val_gen, test_gen
    keras.backend.clear_session()
    gc.collect()
    return result


results = {}

## 6. Baseline: Custom CNN

In [ ]:
def build_cnn():
    model = Sequential([
        Input(shape=IMAGE_SIZE + (3,)),
        Conv2D(32, (3, 3), activation="relu"),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation="relu"),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation="relu"),
        MaxPooling2D((2, 2)),
        GlobalAveragePooling2D(),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation="softmax")
    ])
    compile_model(model, 1e-3)
    return model

cnn_train_gen, cnn_val_gen, cnn_test_gen = create_generators(lambda x: x / 255.0)
cnn_model = build_cnn()
cnn_model.summary()

In [ ]:
start = time.time()
cnn_history = cnn_model.fit(
    cnn_train_gen,
    validation_data=cnn_val_gen,
    epochs=CNN_EPOCHS,
    callbacks=make_callbacks("CNN", patience=8),
    verbose=1
)
cnn_time = time.time() - start

In [ ]:
results["CNN"] = evaluate_model("CNN", cnn_model, cnn_test_gen, cnn_history.history, cnn_time)
plot_result(results["CNN"])

del cnn_model, cnn_train_gen, cnn_val_gen, cnn_test_gen
keras.backend.clear_session()
gc.collect()

## 7. Transfer Learning Models

Each pretrained model is trained in two phases:

1. **Feature extraction:** the ImageNet backbone is frozen and only the new classification head is trained.
2. **Fine-tuning:** the last 30 backbone layers are unfrozen (BatchNorm layers stay frozen) and trained with a very low learning rate.

### 7.1 EfficientNetB0

In [ ]:
results["EfficientNetB0"] = run_transfer_experiment("EfficientNetB0", EfficientNetB0, efficientnet_preprocess)

In [ ]:
plot_result(results["EfficientNetB0"])

### 7.2 ResNet50

In [ ]:
results["ResNet50"] = run_transfer_experiment("ResNet50", ResNet50, resnet_preprocess)

In [ ]:
plot_result(results["ResNet50"])

### 7.3 MobileNetV2

In [ ]:
results["MobileNetV2"] = run_transfer_experiment("MobileNetV2", MobileNetV2, mobilenet_preprocess)

In [ ]:
plot_result(results["MobileNetV2"])

### 7.4 DenseNet121

In [ ]:
results["DenseNet121"] = run_transfer_experiment("DenseNet121", DenseNet121, densenet_preprocess)

In [ ]:
plot_result(results["DenseNet121"])

## 8. Final Comparison

### 8.1 Summary Table

In [ ]:
summary_df = pd.DataFrame([
    {
        "Model": r["name"],
        "Test Accuracy": r["test_accuracy"],
        "Best Val Accuracy": r["best_val_accuracy"],
        "Macro Precision": r["macro_precision"],
        "Macro Recall": r["macro_recall"],
        "Macro F1": r["macro_f1"],
        "Parameters (M)": r["parameters"] / 1e6,
        "Training Time (min)": r["train_time_min"]
    }
    for r in results.values()
]).sort_values("Test Accuracy", ascending=False).reset_index(drop=True)

summary_df.to_csv(os.path.join(WORKING_DIR, "model_comparison.csv"), index=False)
summary_df.style.format({
    "Test Accuracy": "{:.4f}", "Best Val Accuracy": "{:.4f}",
    "Macro Precision": "{:.4f}", "Macro Recall": "{:.4f}", "Macro F1": "{:.4f}",
    "Parameters (M)": "{:.2f}", "Training Time (min)": "{:.1f}"
}).background_gradient(subset=["Test Accuracy", "Macro F1"], cmap="Greens")

### 8.2 Test Accuracy and Macro F1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, metric in zip(axes, ["Test Accuracy", "Macro F1"]):
    sns.barplot(data=summary_df, x="Model", y=metric, palette="viridis", ax=ax)
    ax.set_ylim(max(0, summary_df[metric].min() - 0.05), 1.0)
    ax.set_title(f"{metric} by Model")
    ax.tick_params(axis="x", rotation=20)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.4f", padding=3)

plt.tight_layout()
plt.show()

### 8.3 Training Curves of All Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for name, r in results.items():
    epochs = range(1, len(r["history"]["val_accuracy"]) + 1)
    axes[0].plot(epochs, r["history"]["val_accuracy"], label=name)
    axes[1].plot(epochs, r["history"]["val_loss"], label=name)

axes[0].set_title("Validation Accuracy Comparison")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Validation Accuracy")
axes[1].set_title("Validation Loss Comparison")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation Loss")
axes[0].legend()
axes[1].legend()

plt.tight_layout()
plt.show()

### 8.4 Per-Class F1 Score

In [ ]:
f1_df = pd.DataFrame({name: r["per_class_f1"] for name, r in results.items()}).T

plt.figure(figsize=(9, 5))
sns.heatmap(f1_df, annot=True, fmt=".3f", cmap="YlGnBu", vmin=f1_df.values.min() - 0.02, vmax=1.0)
plt.title("Per-Class F1 Score by Model")
plt.ylabel("Model")
plt.xlabel("Class")
plt.tight_layout()
plt.show()

### 8.5 Confusion Matrices Side by Side

In [ ]:
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4.5))

for ax, (name, r) in zip(axes, results.items()):
    sns.heatmap(
        r["confusion_matrix"], annot=True, fmt="d", cmap="Blues", cbar=False,
        xticklabels=r["class_names"], yticklabels=r["class_names"], ax=ax
    )
    ax.set_title(f"{name}\nAcc: {r['test_accuracy']:.4f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 8.6 Accuracy vs Model Size and Training Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].scatter(summary_df["Parameters (M)"], summary_df["Test Accuracy"], s=150, c=range(len(summary_df)), cmap="viridis")
axes[1].scatter(summary_df["Training Time (min)"], summary_df["Test Accuracy"], s=150, c=range(len(summary_df)), cmap="viridis")

for _, row in summary_df.iterrows():
    axes[0].annotate(row["Model"], (row["Parameters (M)"], row["Test Accuracy"]), textcoords="offset points", xytext=(8, 6))
    axes[1].annotate(row["Model"], (row["Training Time (min)"], row["Test Accuracy"]), textcoords="offset points", xytext=(8, 6))

axes[0].set_title("Test Accuracy vs Parameters")
axes[0].set_xlabel("Parameters (Millions)")
axes[0].set_ylabel("Test Accuracy")
axes[1].set_title("Test Accuracy vs Training Time")
axes[1].set_xlabel("Training Time (minutes)")
axes[1].set_ylabel("Test Accuracy")

plt.tight_layout()
plt.show()

### 8.7 Best Model

In [ ]:
best = summary_df.iloc[0]
print(f"Best model: {best['Model']}")
print(f"Test Accuracy: {best['Test Accuracy']:.4f}")
print(f"Macro F1: {best['Macro F1']:.4f}")
print(f"Parameters: {best['Parameters (M)']:.2f}M")
print(f"Training Time: {best['Training Time (min)']:.1f} min")

## 9. Notes

- Saved weights for every model are in `/kaggle/working/<ModelName>_best.keras` and the comparison table is in `/kaggle/working/model_comparison.csv`.
- If the session runs out of GPU memory, lower `BATCH_SIZE` to 16 in the configuration cell.
- To shorten the run, reduce `HEAD_EPOCHS`, `FINE_TUNE_EPOCHS` or `CNN_EPOCHS`.